# Strategic Pricing Analysis: Minimizing Risk In Price Elasticity Testing

## Executive Summary
This project evaluates the financial viability and operational risk of increasing a product's price point from \\$19.99 to \\$24.99.

Unlike a standard A/B test which seeks to find a "winner," this analysis employs Non-Inferiority Testing. The goal was to prove that the higher price point would not cause the conversion rate (CVR) to drop below a critical 8% break-even floor, thereby ensuring the revenue lift justifies the loss in customer volume.

__Objective:__ Evaluate the financial viability of increasing product price from \\$19.99 to \\$24.99.

**The Challenge:** Determine if revenue lift from the higher price justifies any drop in conversion rate from the price increase.

**Outcome:** The test passed the non-inferiority safety check (p_val - 0.0205), projecting a 9.5% monthly revenue lift, despite a thin statistical safety margin.price increase.

## Experimental Design
__Methodology:__ Monaic A/B test (independent groups)

__Primary Metric:__ Conversion rate

__Counter Metric:__ Revenue Per Visitor(RPV)

__Hypothesis:__

&emsp; &emsp; __Null ($H_0$):__ Conversion will drop below the 8% break-even floor.

&emsp; &emsp; __Alternative ($H_a$):__ Conversion will stay above the break-even floor (Non-inferiority)

__Sample Size:__ Calculated minimum $N=2524$ per group; Actual $N=5000$ per group used to increase presicion.

## Implementation

#### Calculate required sample size for alpha 0.05 and power 80%:

In [1]:
# Calculate required sample size for an alpha of 0.05 and a power of 80%

import statsmodels.stats.api as sms
from statsmodels.stats.power import NormalIndPower

# Define safety floor (Product conversion baseline is 10%, break-even is 8%. Safety floor is 2% below baseline)
baseline = 0.1
safety_floor = baseline - 0.02

# Define effect size for non-inferiority safety
h_effect = sms.proportion_effectsize(baseline, safety_floor)

# Define alpha
alpha = 0.05

print(f'Product baseline: {baseline}, Safety floor: {safety_floor}, Effect size: {h_effect}')

# Define n per group (one-sided since we are looking to refute a drop)
n = NormalIndPower().solve_power(effect_size=h_effect, alpha=alpha, power=0.8, alternative='larger')
print(f'Required sample size per group: {int(n)}')

Product baseline: 0.1, Safety floor: 0.08, Effect size: 0.0699880043701876
Required sample size per group: 2524


#### Import sample data file:

In [2]:
# Import sample data file
import pandas as pd
import numpy as np

sim_ab = pd.read_csv('simulated_ab.csv')
print(sim_ab.head())
print(sim_ab.shape)

# Add group variable
sim_ab['group'] = np.where(sim_ab['price'] == 19.99, 'control', 'variant')
print(sim_ab.head())

# Count data split
print('\nControl and variant sample split: \n', sim_ab.groupby('group')['price'].count())

c_count, v_count = sim_ab['group'].value_counts().tolist()
c_converted, v_converted = sim_ab.groupby('group')['converted'].sum().tolist()
print(f'\nControl conversions: {c_converted}, Variant conversions: {v_converted}')
print(f'\nConversion rate for control sample: {round(c_converted/c_count, 4)}')
print(f'Conversion rate for variant sample: {round(v_converted/v_count, 4)}')

   user_id  price  converted
0        0  19.99          0
1        1  19.99          1
2        2  19.99          0
3        3  19.99          0
4        4  19.99          0
(10000, 3)
   user_id  price  converted    group
0        0  19.99          0  control
1        1  19.99          1  control
2        2  19.99          0  control
3        3  19.99          0  control
4        4  19.99          0  control

Control and variant sample split: 
 group
control    5000
variant    5000
Name: price, dtype: int64

Control conversions: 479, Variant conversions: 438

Conversion rate for control sample: 0.0958
Conversion rate for variant sample: 0.0876


#### Available sample size is more than sufficient (using all 5000 samples per group to increase presicion). Proceed with p-value calculation (using proportions_ztest to focus on one-sided value):

In [3]:
# Available sample size is sufficient
# Calculate p-value (use proportions_ztest to focus on one-sided value)

from statsmodels.stats.proportion import proportions_ztest

count = np.array([v_converted, c_converted])
total = np.array([v_count, c_count])

# Define non-inferiority margin (allowable drop), which has been decided to be 2%
margin = -0.02

# p-value
z_stat, p_value = proportions_ztest(count, total, value=margin, alternative='larger')

print(f'P-value: {p_value:.4f}\n')
if p_value < alpha:
    print('Reject Null: The price change is safe.')
else:
    print(f'Fail To Reject: Cannot prove safety. The risk of a {(margin * -100):.0f}% drop or more is high')

P-value: 0.0205

Reject Null: The price change is safe.


#### Validate impact and quality of results:

In [4]:
# Validate results
from statsmodels.stats.proportion import confint_proportions_2indep
import scipy.stats as stats

# Check confidence interval, and confirm that our margin is below the lower bound
ci_lower, ci_upper = confint_proportions_2indep(v_converted, v_count, c_converted, c_count, method='wald', compare='diff')
print(f'Lower confidence interval: {ci_lower}, Upper confidence interval: {ci_upper}')

# Simple Ratio Mismatch(SRM)
chi2, srm_p_val = stats.chisquare([v_count, c_count], f_exp=[(v_count + c_count)/2]*2)
print(f'\nSimple ratio mismatch: {srm_p_val}')

# Post-test power
# h_observed = sms.proportion_effectsize(round(v_converted/v_count, 2), safety_floor)
post_power = NormalIndPower().solve_power(effect_size=h_effect, nobs1=v_count, alpha=alpha, ratio=1.0, alternative='larger')
print(f'\nPost-hoc power: {post_power}')

# Revenue per visitor (RPV)
# Control price = 19.99, Variant price = 24.99
c_rpv = (c_converted * 19.99) / c_count
v_rpv = (v_converted * 24.99) / v_count
print(f'\nControl RPV: {c_rpv}, Variant RPV: {v_rpv}, RPV increase: {(v_rpv-c_rpv)/c_rpv}')

Lower confidence interval: -0.01951184855400971, Upper confidence interval: 0.0031118485540097118

Simple ratio mismatch: 1.0

Post-hoc power: 0.9681694995252323

Control RPV: 1.915042, Variant RPV: 2.1891239999999996, RPV increase: 0.14312062085322397


#### Visualize results and impact:

In [ ]:
import matplotlib.pyplot as plt

# 1. Safety Margin
plt.figure(figsize=(8, 5))
labels = ['Control ($19.99)', 'Treatment ($24.99)']
rates = [0.10, 0.0876]
errors = [0, 0.0071] # Simplified CI margin
plt.bar(labels, rates, yerr=errors, capsize=10, color=['gray', 'blue'], alpha=0.7)
plt.axhline(y=0.08, color='red', linestyle='--', label='Break-even Floor (8%)')
plt.title('Conversion Rate vs. Critical Safety Floor')
plt.ylabel('Conversion Rate')
plt.legend()
plt.show()

# 2. Revenue Comparison
revenue = [1.99, 2.19] # RPV
plt.figure(figsize=(8, 5))
plt.bar(labels, revenue, color=['gray', 'green'], alpha=0.7)
plt.title('Revenue Per Visitor (RPV) Comparison')
plt.ylabel('USD ($)')
plt.show() 

# 3. Volume vs. Value
labels = ['Control ($19.99)', 'Variant ($24.99)']
conversions = [500, 438] # Volume
revenue = [9995.00, 10945.62] # Value (based on 5000 users each)
x = np.arange(len(labels))
width = 0.35
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Volume (Conversions)
color = 'tab:blue'
ax1.set_ylabel('Total Conversions (Volume)', color=color, fontweight='bold')
bar1 = ax1.bar(x - width/2, conversions, width, label='Conversions', color=color, alpha=0.6)
ax1.tick_params(axis='y', labelcolor=color)

# Create a second axis for Value (Revenue)
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Total Revenue (Value)', color=color, fontweight='bold')
bar2 = ax2.bar(x + width/2, revenue, width, label='Revenue', color=color, alpha=0.6)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Volume (Total Conversions) vs. Value (Total revenue)', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)

# Add a legend
fig.tight_layout()
plt.show()

## Key Results and Analysis

__Statistical Significance:__ p_val = 0.0205. P_value is less than our set alpha of 0.05, thus we can reject the null hypothesis. The new price is statistically safe and non-inferior (ie above our break-even floor).

__Data Integrity:__ SRM p_value = 1.0. Ramdomization was perfect, ensuring no selection bias.

__Safety Margin Analysis:__ The lower bound of the 95% confidence interval is -0.0195. This is a critical insight showing that the conversion rate at the new price could realistically be as low as 8.05%, which is close to our break-even point of 8%.

__Interpretation:__ Although we are in the safe zone, our matrgin of error is slim (<0.1), so implementation needs to be monitored closely.

## Business Impact Projection

__Control Revenue:__ $19.99 * 0.1 = $1.99 RPV

__Variant Revenue:__ $24.99 * 0.0876 = $2.19 RPV

__Projected Lift:__ +$1901.24 (+9.51%) per 10,000 visistors

## Final Recommendation: Guarded Rollout

Based on the thin safety margin, I recommend a Guarded Rollout rather than a full 100% launch:

__Phased Deployment:__ Launch to 75% of traffic to maintain a control group "safety net" for the first 30 days.

__LTV Monitoring:__ Analyze if the 12.4% lost customer volume represents high-value long-term users or one-time shoppers.

__Automated Kill-Switch:__ Implement a real-time monitor to trigger an automatic rollback if CVR dips below the 8.0% threshold.
